# Description

In this notebook, I will load the generated code from HumanEval (by GPT4) and get the code embedding

In [1]:
import os 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch 
import transformers
from transformers import AutoTokenizer, AutoModel

from utils.evaluation_human_eval import *

In [2]:
print(torch.__version__)    
print(torch.version.cuda)
print(torch.cuda.nccl.version())
print(transformers.__version__)

2.6.0+cu124
12.4
(2, 21, 5)
4.45.2


# 1. Load code embedding

In [3]:
# Load model + tokenizer
model_name = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [4]:
def embed_code(code_snippet: str):
    # Tokenize input
    inputs = tokenizer(code_snippet, return_tensors="pt", truncation=True, padding=True)
    
    # Get model outputs
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Use CLS token embedding as the vector
    embeddings = outputs.last_hidden_state[:, 0, :]
    return embeddings.squeeze().numpy()

In [5]:
code = """
def add(a, b):
    return a + b
"""

vec = embed_code(code)
print("Embedding shape:", vec.shape)

Embedding shape: (768,)


# 2. Generate Embeding for Human Eval dataset

In [6]:
PATH_GENERATED_DATA_FOLDER = "data/generated"


In [7]:
list_files = os.listdir(PATH_GENERATED_DATA_FOLDER)
list_csv_files = [f for f in list_files if f.endswith('.csv')]
print(f"List of generated CSV files: ")
list_csv_files

List of generated CSV files: 


['human_eval_generated_deepseek.csv',
 'human_eval_generated_gpt4o.csv',
 'human_eval_generated_qwen_TDD.csv',
 'human_eval_generated_deepseek_COT.csv',
 'human_eval_generated_qwen.csv',
 'human_eval_generated_deepseek_TDD.csv',
 'human_eval_generated_qwen_COT.csv',
 'human_eval_generated_gpt4o_COT.csv']

In [19]:
df = pd.DataFrame()
for file in list_csv_files:
    current_df = pd.read_csv(os.path.join(PATH_GENERATED_DATA_FOLDER, file))
    llm_name = file.split('_')[3]
    print(f"{file}: {current_df.shape[0]} rows, {current_df.shape[1]} columns")
    current_df['llm_name'] = llm_name
    df = pd.concat([df, current_df], ignore_index=True)

print()
df.drop_duplicates(subset=['generated_code', 'test_case'], inplace=True)
df = df.reset_index(drop=True)
print("Combined dataframe shape:", df.shape)
df.sample()

human_eval_generated_deepseek.csv: 163 rows, 7 columns
human_eval_generated_gpt4o.csv: 164 rows, 7 columns
human_eval_generated_qwen_TDD.csv: 164 rows, 7 columns
human_eval_generated_deepseek_COT.csv: 164 rows, 7 columns
human_eval_generated_qwen.csv: 163 rows, 7 columns
human_eval_generated_deepseek_TDD.csv: 161 rows, 7 columns
human_eval_generated_qwen_COT.csv: 164 rows, 8 columns
human_eval_generated_gpt4o_COT.csv: 164 rows, 8 columns

Combined dataframe shape: (1048, 9)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage,llm_name,reasoning_text
922,"import math\n\n\ndef poly(xs: list, x: float):...","def find_zero(xs: list):\n def poly(xs, x):...",\n\nMETADATA = {}\n\n\ndef check(candidate):\n...,find_zero,0,0,0.0,gpt4o,1. **What the function must do**: \n The `fi...


In [20]:
df['generated_code_embedd'] = df['generated_code'].apply(embed_code)
df['test_case_embedd'] = df['test_case'].apply(embed_code)

print(f"[INFO] Embedding generation completed!")

[INFO] Embedding generation completed!


## 2.1. Test embedding space

In [21]:
def distance_2_vector(vec1, vec2, type="l2"):
    if type == "l2":
        return np.linalg.norm(vec1 - vec2)
    elif type == "cosine":
        return 1 - np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    else:
        raise ValueError("Unknown distance type")

In [22]:
idx = np.random.randint(0, len(df))

gen_code = df.loc[idx, 'generated_code']
test_case = df.loc[idx, 'test_case']
entry_point = df.loc[idx, 'entry_point']

gen_code_embedd = df.loc[idx, 'generated_code_embedd']
test_case_embedd = df.loc[idx, 'test_case_embedd']

print("Generated Code:\n", gen_code)
print("-"*50)
print("Test Case:\n", test_case)

Generated Code:
 def cycpattern_check(a , b):
    # Check if b is a substring of a
    if b in a:
        return True
    
    # Check if any rotation of b is a substring of a
    for i in range(len(b)):
        if b[i:] + b[:i] in a:
            return True
    
    return False
--------------------------------------------------
Test Case:
 def check(candidate):

    # Check some simple cases
    #assert True, "This prints if this assert fails 1 (good for debugging!)"

    # Check some edge cases that are easy to work out by hand.
    #assert True, "This prints if this assert fails 2 (also good for debugging!)"
    assert  candidate("xyzw","xyw") == False , "test #0"
    assert  candidate("yello","ell") == True , "test #1"
    assert  candidate("whattup","ptut") == False , "test #2"
    assert  candidate("efef","fee") == True , "test #3"
    assert  candidate("abab","aabb") == False , "test #4"
    assert  candidate("winemtt","tinem") == True , "test #5"




In [23]:
result = evaluate_asserts(gen_code, test_case, entry_point)
print(result)

{'total_asserts': 6, 'passed': 6, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


In [24]:
distance = distance_2_vector(gen_code_embedd, test_case_embedd, type="l2")
print(f"L2 Distance between generated code and test case embeddings: {distance}")

L2 Distance between generated code and test case embeddings: 3.5227279663085938


In [25]:
df.sample()

,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage,llm_name,reasoning_text,generated_code_embedd,test_case_embedd
875,"\n\n\ndef sum_squares(lst):\n """"""""\n Thi...",def sum_squares(lst):\n total_sum = 0\n ...,def check(candidate):\n\n # Check some simp...,sum_squares,11,3,0.272727,qwen,No <reasoning> block found,"[-0.2541338, 0.15470229, -0.11691105, 0.211008...","[-0.16156206, -0.016205445, -0.15150249, 0.202..."


## Save file

In [26]:
OUTPUT_CSV_PATH = "data/embedding/human_eval_codebert_embeddings.csv"
df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"[INFO] Saved embeddings to {OUTPUT_CSV_PATH}")

[INFO] Saved embeddings to data/embedding/human_eval_codebert_embeddings.csv
